# Earth Risk Watch: end-to-end cloud pilot

This notebook rebuilds the pilot in a temporary Google Colab runtime. Raw and generated data stay out of Git. The final cell downloads a compact ZIP of products and provenance.

In [ ]:
!git clone -q https://github.com/winecube-jr/earth-risk-watch.git
%cd earth-risk-watch
%pip install -q -e '.[geo,cloud,notebook]'

## Authenticate Earth Engine
Authentication is interactive and remains in the temporary Colab runtime. Never paste or commit service-account keys.

In [ ]:
import ee

EARTHENGINE_PROJECT = "earth-risk-watch"  # @param {type:'string'}
ee.Authenticate()
ee.Initialize(project=EARTHENGINE_PROJECT)

## Build bounded source and feature products
The LiDAR request is server-resampled to 10 m and limited to the pilot bounding box.

In [ ]:
import os
import subprocess

os.environ["EARTHENGINE_PROJECT"] = EARTHENGINE_PROJECT
commands = [
    ["earth-risk", "extract-ea-pilot"],
    ["earth-risk", "extract-ea-geometry"],
    ["earth-risk", "build-grid"],
    ["earth-risk", "sentinel-summary"],
    ["earth-risk", "sentinel-seasonal"],
    ["earth-risk", "sentinel-grid-features"],
    ["earth-risk", "build-ecological-outcomes"],
    ["earth-risk", "extract-water-sampling-points"],
    ["earth-risk", "extract-water-observations"],
    ["earth-risk", "extract-lidar-terrain"],
    ["earth-risk", "build-terrain-features"],
    ["earth-risk", "build-monitoring-features"],
    ["earth-risk", "build-risk-screen"],
    ["earth-risk", "publish-risk-map"],
    ["earth-risk", "build-evidence-pack"],
]
for command in commands:
    print(">", " ".join(command), flush=True)
    subprocess.run(command, check=True)

In [ ]:
import geopandas as gpd
from IPython.display import display

screen = gpd.read_file("data/products/risk/pilot-screen.geojson")
display(
    screen[
        [
            "cell_id",
            "screening_score",
            "screening_band",
            "top_quintile_frequency",
            "monitoring_distance_km",
        ]
    ].nlargest(15, "screening_score")
)

## Download compact results
The native LiDAR raster is deliberately excluded from the ZIP; it can always be reproduced from its WCS request and provenance.

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/earth-risk-watch-products", "zip", "data/products")
files.download("/content/earth-risk-watch-products.zip")